In [ ]:
from dataclasses import dataclass


@dataclass
class Person:
    """Group member. Convention: balance > 0 receives, balance < 0 owes."""
    id: int
    balance: int

    @property
    def is_settled(self) -> bool:
        return self.balance == 0

In [ ]:
class Creditor(Person):
    """Person with a positive balance: has money to receive `balance`."""

    def receive(self, amount: int) -> None:
        if not 0 < amount <= self.balance:
            raise ValueError(f"Creditor {self.id}: invalid amount {amount} to receive. Balance is {self.balance}")
        self.balance -= amount


In [ ]:
class Debtor(Person):
    """Person with a negative balance: owes `debt` (= -balance)."""

    @property
    def debt(self) -> int:
        return -self.balance

    def pay(self, amount: int) -> None:
        if not 0 < amount <= self.debt:
            raise ValueError(f"Debtor {self.id}: invalid amount {amount} to pay. Debt is {self.debt}")
        self.balance += amount

In [ ]:
@dataclass(frozen=True)
class Exchange:
    payer: Debtor
    receiver: Creditor
    amount: int

    def __str__(self) -> str:
        return f"{self.payer.id} pays {self.amount} to {self.receiver.id}"

In [ ]:
from typing import Self


class BalancePool:
    """Pool of people with balance != 0. Automatically remove those who settle."""

    def __init__(self, creditors: list[Creditor], debtors: list[Debtor]):
        self._creditors = creditors
        self._debtors = debtors

    @classmethod
    def from_balances(cls, balances: list[int]) -> Self:
        if sum(balances) != 0:
            raise ValueError("Sum of balances must be equal to 0")

        creditors = [Creditor(i, b) for i, b in enumerate(balances) if b > 0]
        debtors = [Debtor(i, b) for i, b in enumerate(balances) if b < 0]
        return cls(creditors, debtors)

    @property
    def people(self) -> list[Person]:
        return [*self._creditors, *self._debtors]

    @property
    def creditors(self) -> list[Creditor]:
        return list(self._creditors)

    @property
    def debtors(self) -> list[Debtor]:
        return list(self._debtors)

    def largest_creditor(self) -> Creditor:
        return max(self._creditors, key=lambda c: c.balance)

    def largest_debtor(self) -> Debtor:
        return max(self._debtors, key=lambda d: d.debt)

    def apply(self, exchange: Exchange) -> None:
        exchange.payer.pay(exchange.amount)
        exchange.receiver.receive(exchange.amount)

        if exchange.payer.is_settled:
            self._debtors.remove(exchange.payer)
        if exchange.receiver.is_settled:
            self._creditors.remove(exchange.receiver)

    def all_debtors_are_settled(self) -> bool:
        return len(self._debtors) == 0

    def __repr__(self) -> str:
        return f"BalancePool(creditors={self._creditors}, debtors={self._debtors})"

    def top_creditors(self, k: int) -> list[Creditor]:
        """The k largest creditors (by balance), from largest to smallest"""
        return sorted(self._creditors, key=lambda c: c.balance, reverse=True)[:k]

    def top_debtors(self, k: int) -> list[Debtor]:
        """The k largest debtors (by debt), from largest to smallest."""
        return sorted(self._debtors, key=lambda d: d.debt, reverse=True)[:k]

In [ ]:
from pathlib import Path


def read_balances_from(file: Path) -> BalancePool:
    """Load balances from a file and return them as a balance pool."""
    with open(file, "r") as f:
        balances = []

        for line in f:
            parts = line.strip().split()
            balance = int(parts[-1])
            balances.append(balance)

        return BalancePool.from_balances(balances)

In [ ]:
instance_path = Path("instances/instancia_splitwise_50_8.txt")
balances = read_balances_from(instance_path)
balances

In [ ]:
def greedy_constructive(balances: BalancePool) -> list[Exchange]:
    exchanges = []

    while not balances.all_debtors_are_settled():
        exchanges += _settle_exact_matches(balances)

        if balances.all_debtors_are_settled():
            break

        exchange = _pick_greedy_exchange(balances)
        balances.apply(exchange)
        exchanges.append(exchange)

    return exchanges


def _settle_exact_matches(balances: BalancePool) -> list[Exchange]:
    exchanges = []

    for creditor in balances.creditors:
        for debtor in balances.debtors:
            if creditor.balance == debtor.debt:
                exchange = Exchange(debtor, creditor, creditor.balance)
                balances.apply(exchange)
                exchanges.append(exchange)
                break  # this creditor is already settled

    return exchanges


def _pick_greedy_exchange(balances: BalancePool) -> Exchange:
    largest_debtor = balances.largest_debtor()
    largest_creditor = balances.largest_creditor()
    amount = min(largest_debtor.debt, largest_creditor.balance)
    return Exchange(largest_debtor, largest_creditor, amount)


In [ ]:
import copy

greedy_solution = greedy_constructive(copy.deepcopy(balances))
greedy_solution, len(greedy_solution)

In [ ]:
import random
import math


def randomized_constructive(
    balances: BalancePool, alpha: float = 0.1, random_seed: int | None = None
) -> list[Exchange]:
    if not 0 <= alpha <= 1:
        raise ValueError("alpha should be between 0 and 1")

    rng = random.Random(random_seed)
    exchanges = []

    while not balances.all_debtors_are_settled():
        top_creditors, top_debtors = _build_restricted_candidates_lists(balances, alpha)

        exchange = _pick_random_exchange(top_creditors, top_debtors, rng)

        balances.apply(exchange)
        exchanges.append(exchange)

    return exchanges


def _build_restricted_candidates_lists(
    balances: BalancePool, alpha: float
) -> tuple[list[Creditor], list[Debtor]]:
    k_creditors = max(1, math.ceil(alpha * len(balances.creditors)))
    k_debtors = max(1, math.ceil(alpha * len(balances.debtors)))
    top_creditors = balances.top_creditors(k_creditors)
    top_debtors = balances.top_debtors(k_debtors)
    return top_creditors, top_debtors


def _pick_random_exchange(
    candidate_creditors: list[Creditor],
    candidate_debtors: list[Debtor],
    rng: random.Random
) -> Exchange:
    creditor = rng.choice(candidate_creditors)
    debtor = rng.choice(candidate_debtors)
    amount = min(debtor.debt, creditor.balance)
    return Exchange(debtor, creditor, amount)

In [ ]:
randomized_solution = randomized_constructive(copy.deepcopy(balances))
randomized_solution, len(randomized_solution)